# СТАТИСТИЧЕСКИЙ АНАЛИЗ ДАННЫХ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pymorphy3

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, mean_squared_error, r2_score)
from sklearn.pipeline import Pipeline

from scipy import stats
from scipy.stats import shapiro, kruskal, mannwhitneyu, spearmanr

# Загрузка необходимых ресурсов NLTK
nltk.download('punkt')
nltk.download('stopwords')

ЗАГРУЗКА ДАННЫХ

In [ ]:
df_1 = pd.read_excel('data/anketolog_data.xlsx')
df_2 = pd.read_excel('data/survey_data.xlsx')

# Словарь для переименования столбцов
index_to_name = {
    0: 'Id', 1: 'gender', 2: 'grandiosity', 3: 'fantasy_absorbtion',
    4: 'unique_belief', 5: 'attention_seek', 6: 'spec_treat',
    7: 'manipulation', 8: 'empathy_lack', 9: 'envy', 10: 'arrogance',
    11: 'auth1', 12: 'auth2', 13: 'auth3', 14: 'auth4', 15: 'auth5',
    16: 'auth6', 17: 'auth7', 18: 'auth8', 19: 'self_suf1', 20: 'self_suf2',
    21: 'self_suf3', 22: 'self_suf4', 23: 'self_suf5', 24: 'self_suf6',
    25: 'superiority1', 26: 'superiority2', 27: 'superiority3',
    28: 'superiority4', 29: 'superiority5', 30: 'show_off1', 31: 'show_off2',
    32: 'show_off3', 33: 'show_off4', 34: 'show_off5', 35: 'show_off6',
    36: 'show_off7', 37: 'people_use1', 38: 'people_use2', 39: 'people_use3',
    40: 'people_use4', 41: 'people_use5', 42: 'vanity1', 43: 'vanity2',
    44: 'vanity3', 45: 'uniqueness1', 46: 'uniqueness2', 47: 'uniqueness3',
    48: 'uniqueness4', 49: 'uniqueness5', 50: 'uniqueness6'
}

# Переименование и объединение
df_1.rename(columns={df_1.columns[i]: new_name for i, new_name in index_to_name.items()}, inplace=True)
df_2.rename(columns={df_2.columns[i]: new_name for i, new_name in index_to_name.items()}, inplace=True)
data = pd.concat([df_1, df_2], axis=0, ignore_index=True)

БИНАРИЗАЦИЯ ОТВЕТОВ NPI-40 (1 - нарциссическое уотвеждение, 0 - обычное)

In [ ]:
def binarize_multiple_columns(df, mapping_dict):
    """Заменяет значения в указанных столбцах на 1 или 0 по заданному словарю."""
    def map_func(x, positive_text):
        return 1 if x == positive_text else 0
    for col_idx, pos_text in mapping_dict.items():
        df.iloc[:, col_idx] = df.iloc[:, col_idx].apply(lambda x: map_func(x, pos_text))
    return df

mapping = {
    11: 'Я бы предпочел быть лидером',
    12: 'Я вижу себя хорошим лидером',
    13: 'Я добьюсь успеха',
    14: 'Люди, похоже, всегда признают мой авторитет',
    15: 'У меня природный талант влиять на людей',
    16: 'Я уверен в себе',
    17: 'Мне нравится обладать властью над людьми',
    18: 'Я - прирождённый лидер',
    19: 'Я редко когда завишу от кого-то другого, чтобы получить результат',
    20: 'Я люблю брать на себя ответственность за принятие решений',
    21: 'Я более способный, нежели другие люди',
    22: 'Я могу жить так, как я хочу',
    23: 'Я всегда знаю, что я делаю',
    24: 'Я собираюсь стать великим человеком',
    25: 'Я экстраординарная личность',
    26: 'Я знаю, что я хороший, потому что все то и дело мне об этом говорят',
    27: 'Мне нравится, когда мне делают комплименты',
    28: 'Я думаю, я особенный человек',
    29: 'Я хочу, чтобы кто-нибудь когда-нибудь написал мою биографию',
    30: 'Я склонен выставлять себя напоказ, если есть такая возможность',
    31: 'Скромность мне не идет',
    32: 'Я расстраиваюсь, когда люди не замечают, как я выгляжу, когда я выхожу на публику',
    33: 'Мне нравится быть в центре внимания',
    34: 'Мне нравится быть источником новых мод и увлечений',
    35: 'Мне на самом деле нравится быть в центре внимания',
    36: 'Я сделаю всё что угодно на спор',
    37: 'Я могу читать человека, словно книгу',
    38: 'Я могу заставить любого поверить во всё, во что захочу',
    39: 'Мне легко манипулировать людьми',
    40: 'Обычно я могу выкрутиться в любой ситуации',
    41: 'Всем нравится слушать мои истории',
    42: 'Я люблю смотреть на своё тело',
    43: 'Мне нравится смотреть на себя в зеркало',
    44: 'Мне нравится демонстрировать моё тело',
    45: 'Я никогда не удовлетворяюсь, пока не получу всё, чего я заслуживаю',
    46: 'Я много ожидаю от других людей',
    47: 'Я хочу в мире что-то из себя представлять',
    48: 'У меня сильное стремление к власти',
    49: 'Я настаиваю на том, чтобы получать должное уважение',
    50: 'Если бы я правил миром, он был бы гораздо лучшим местом'
}

df = binarize_multiple_columns(data, mapping)

ПОДСЧЁТ БАЛЛОВ ПО ЧЕРТАМ И УРОВНЯМ НАРЦИССИЗМА

In [ ]:
feature_ranges = {
    'authority': range(11, 19),
    'self_suf': range(19, 25),
    'superiority': range(25, 30),
    'show_off': range(30, 37),
    'people_use': range(37, 42),
    'vanity': range(42, 45),
    'uniqueness': range(45, 51)
}

for feature_name, cols in feature_ranges.items():
    df[feature_name] = df.iloc[:, cols].sum(axis=1)

features = ['authority', 'self_suf', 'superiority', 'show_off', 'people_use', 'vanity', 'uniqueness']
df['narcissism'] = df[features].sum(axis=1)

# Оставляем только нужные столбцы
cols_to_keep = ['Id', 'gender'] + \
               [f for f in index_to_name.values() if f not in ['Id', 'gender'] and not f.startswith('auth') and not f.startswith('self_suf') and not f.startswith('superiority') and not f.startswith('show_off') and not f.startswith('people_use') and not f.startswith('vanity') and not f.startswith('uniqueness')] + \
               features + ['narcissism']
df_ready = df[cols_to_keep].copy()

# Категоризация целевой переменной
def map_narcissism_class(value):
    if 0 <= value <= 9:
        return 0  # Низкий уровень
    elif 10 <= value <= 22:
        return 1  # Средний уровень
    else:
        return 2  # Высокий уровень

df_ready['narcissism_class'] = df_ready['narcissism'].astype(int).apply(map_narcissism_class)

СТАТИСТИЧЕСКИЙ АНАЛИЗ

In [ ]:
numerical_features = ['authority', 'self_suf', 'superiority', 'show_off',
                      'people_use', 'vanity', 'uniqueness']
target = 'narcissism_class'

Распределение целевой переменной

In [ ]:
class_order = [0, 1, 2]
class_counts = df_ready['narcissism_class'].value_counts().reindex(class_order, fill_value=0)

plt.figure(figsize=(8, 5))
class_counts.plot(kind='bar', color=['lightgreen', 'skyblue', 'salmon'],
                  edgecolor='black', linewidth=1.2)
plt.title('Распределение классов нарциссизма')
plt.xlabel('Класс нарциссизма (0=низкий, 1=средний, 2=высокий)')
plt.ylabel('Количество респондентов')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=300)
plt.show()
print("Распределение классов:")
print(class_counts)

Описательные статистики и тест на нормальность

In [ ]:
print('\nDescriptive Statistics:')
print(df_ready[numerical_features].describe())

print('\nNormality Tests (Shapiro-Wilk, alpha=0.05):')
for col in numerical_features:
    clean_data = df_ready[col].dropna()
    if 3 < len(clean_data) < 5000:
        stat, p = shapiro(clean_data)
        normal = 'Yes' if p > 0.05 else 'No'
        print(f"{col}: W={stat:.4f}, p={p:.4f}, Normal={normal}")
    else:
        print(f"{col}: Invalid sample size")

# Гистограммы распределения числовых признаков
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()
for idx, col in enumerate(numerical_features):
    axes[idx].hist(df_ready[col].dropna(), bins=20, alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'Frequency of {col} scores')
    axes[idx].set_xlabel(f'{col} values (scores)')
    axes[idx].set_ylabel('Frequency (count of respondents)')
plt.tight_layout()
plt.savefig('distributions_frequency.png', dpi=300)
plt.show()

Boxplots и критерий Краскела-Уоллиса

In [ ]:
print('\nStatistical significance (Kruskal-Wallis test):')
kw_results = {}
for col in numerical_features:
    groups = [df_ready[df_ready['narcissism_class'] == i][col].dropna()
              for i in sorted(df_ready['narcissism_class'].unique())]
    if all(len(g) > 5 for g in groups):
        stat, p = kruskal(*groups)
        kw_results[col] = {'H-stat': round(stat, 3), 'p-value': f'{p:.3e}',
                           'Significant': 'Yes' if p < 0.05 else 'No'}
kw_df = pd.DataFrame(kw_results).T
print(kw_df)

Матрица корреляций Спирмена

In [ ]:
print('\nSpearman Correlation Matrix:')
corr_matrix = df_ready[numerical_features].corr(method='spearman')
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Spearman Correlation Matrix (non-normal data)')
plt.tight_layout()
plt.savefig('correlation_spearman.png', dpi=300)
plt.show()

TF-IDF анализ текстовых данных

In [ ]:
morph = pymorphy3.MorphAnalyzer()
stop_words = set(stopwords.words('russian'))
USELESS_WORDS = {'это', 'быть', 'мочь', 'такой', 'весь', 'сам', 'она', 'тот', 'кто',
                  'как', 'все', 'еще', 'уже', 'тоже', 'даже', 'лишь', 'через', 'перед',
                  'во', 'на', 'в', 'с', 'у', 'к', 'до', 'по', 'за', 'над', 'под',
                  'при', 'для', 'без', 'из', 'от', 'со', 'то', 'те', 'им'}

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^а-яё\s]', ' ', text)
    text = re.sub(r'\d+', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    lemmas = []
    for word in words:
        parsed = morph.parse(word)[0]
        normal_form = parsed.normal_form
        if normal_form not in USELESS_WORDS and len(normal_form) > 2:
            lemmas.append(normal_form)
    return ' '.join(lemmas)

text_features = ['grandiosity', 'fantasy_absorbtion', 'unique_belief', 'attention_seek',
                'spec_treat', 'manipulation', 'empathy_lack', 'envy', 'arrogance']

# Очистка текстов
for feature in text_features:
    df_ready[f'cleaned_{feature}'] = df_ready[feature].apply(preprocess_text)
    print(f'cleaned_{feature}: {df_ready[f"cleaned_{feature}"].str.len().mean():.0f} символов в среднем')

# TF-IDF по блокам
print('\nTF-IDF анализ по блокам...')
tfidf_results = {}
fig, axes = plt.subplots(3, 3, figsize=(20, 18))
axes = axes.ravel()
for idx, feature in enumerate(text_features):
    texts = df_ready[f'cleaned_{feature}'].fillna('')
    vectorizer = TfidfVectorizer(max_features=20, min_df=2, lowercase=False)
    tfidf_matrix = vectorizer.fit_transform(texts)
    mean_tfidf = np.mean(tfidf_matrix.toarray(), axis=0)
    feature_names = vectorizer.get_feature_names_out()
    top_indices = mean_tfidf.argsort()[-10:][::-1]
    top_words = [(feature_names[i], mean_tfidf[i]) for i in top_indices]
    tfidf_results[feature] = top_words
    words = [w[0] for w in top_words]
    scores = [w[1] for w in top_words]
    axes[idx].barh(words, scores, color='skyblue', alpha=0.8)
    axes[idx].set_title(f'Top-10 TF-IDF: {feature}', fontsize=12, pad=10)
    axes[idx].set_xlabel('TF-IDF score')
    axes[idx].set_ylabel('Words')
plt.tight_layout()
plt.savefig('tfidf_by_text_blocks.png', dpi=300)
plt.show()

СОХРАНЕНИЕ ИТОГОВЫХ ДАННЫХ

In [ ]:
df_ready.to_csv('data/data.csv', index=False, encoding='utf-8')
print("Данные сохранены в папку 'data/'")